
# Kaş ROI Deepfake Detection — HOG + LBP + SVM

Bu notebook **Google Colab'da doğrudan çalıştırılmak** üzere hazırlanmıştır.

**Girdi:**
`/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş`

**Kaynak seçim metadatası:**
`/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv`

**Çıktı:**
`/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/<run_id>/`

Ana pipeline:

`Eyebrow ROI -> Grayscale + Resize -> HOG + LBP -> Feature Fusion -> StandardScaler(train only) -> SVM -> Real/Fake`

### Bilimsel ve ekip standardı açısından önemli notlar
- Frame düzeyinde yeniden split **yapılmaz**. Mevcut `train/val/test` ayrımı korunur ve `secim_metadata.csv` üzerinden kaynak video kimliği yeniden oluşturularak split sızıntısı denetlenir.
- SVM hiperparametreleri ve karar threshold'u sadece **validation** setinde seçilir. Test seti yalnızca nihai değerlendirmede açılır.
- Kaş ROI klasörüne hiçbir şey yazılmaz; kaynak veri salt okunur kabul edilir.
- HOG/LBP/SVM klasik makine öğrenmesi olduğu için PyTorch'a özgü `forward/backward`, optimizer ve epoch checkpoint alanları uygulanabilir değildir. Notebook bunu gizlemez; eşdeğer klasik-ML smoke/checkpoint/inference quality gate'leri uygular ve bu durumu kayda geçirir.
- Tüm rapor grafikleri İngilizce, en az 600 px kısa kenarlı üretilir.


In [1]:

# 1) Colab bağımlılıkları
# İlk çalıştırmada güncel paketler kurulur; deney sonunda gerçek ortam `requirements_lock.txt` ile kilitlenir.
!pip -q install scikit-image scikit-learn pyyaml joblib opencv-python-headless


In [2]:

# 2) Google Drive'ı bağla ve merkezi YAML konfigürasyonunu oluştur/yükle
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import yaml

CONFIG_PATH = Path('/content/hog_lbp_svm_eyebrow.yaml')

DEFAULT_CONFIG_YAML = '''seed: 42
region: eyebrow
model_name: hog_lbp_svm

image_width: 128
image_height: 64

hog:
  orientations: 9
  pixels_per_cell: [8, 8]
  cells_per_block: [2, 2]
  block_norm: L2-Hys

lbp:
  points: 8
  radius: 1
  method: uniform

svm:
  class_weight: balanced
  selection_metric: f1
  kernels: [linear, rbf]
  c_values: [0.1, 1.0, 10.0]
  gamma_values: [scale, 0.001, 0.0001]

positive_class: fake
figure_dpi: 150
minimum_figure_short_edge_px: 600
resume_run_id: null

data_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş
roi_metadata_path: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş/metadata.csv
selection_metadata_path: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
results_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar
'''

if not CONFIG_PATH.exists():
    CONFIG_PATH.write_text(DEFAULT_CONFIG_YAML, encoding='utf-8')

try:
    with CONFIG_PATH.open('r', encoding='utf-8') as f:
        CONFIG = yaml.safe_load(f)
except yaml.YAMLError as exc:
    # Önceki/bozuk bir config kaldıysa güvenli şekilde varsayılan config ile yenile.
    backup_path = CONFIG_PATH.with_suffix('.yaml.broken')
    if CONFIG_PATH.exists():
        CONFIG_PATH.replace(backup_path)
    CONFIG_PATH.write_text(DEFAULT_CONFIG_YAML, encoding='utf-8')
    with CONFIG_PATH.open('r', encoding='utf-8') as f:
        CONFIG = yaml.safe_load(f)
    print(f'Bozuk YAML config yenilendi. Eski dosya: {backup_path}')

print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))


Mounted at /content/drive
seed: 42
region: eyebrow
model_name: hog_lbp_svm
image_width: 128
image_height: 64
hog:
  orientations: 9
  pixels_per_cell:
  - 8
  - 8
  cells_per_block:
  - 2
  - 2
  block_norm: L2-Hys
lbp:
  points: 8
  radius: 1
  method: uniform
svm:
  class_weight: balanced
  selection_metric: f1
  kernels:
  - linear
  - rbf
  c_values:
  - 0.1
  - 1.0
  - 10.0
  gamma_values:
  - scale
  - 0.001
  - 0.0001
positive_class: fake
figure_dpi: 150
minimum_figure_short_edge_px: 600
resume_run_id: null
data_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney
  1/Kaş
roi_metadata_path: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney
  1/Kaş/metadata.csv
selection_metadata_path: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney
  1 Frame/secim_metadata.csv
results_root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney
  1/Sonuçlar



In [3]:

# 3) Imports, reproducibility, run klasörleri ve yardımcı fonksiyonlar
import os
import sys
import json
import math
import time
import random
import hashlib
import logging
import platform
import subprocess
from datetime import datetime
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog, local_binary_pattern

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay,
)

SEED = int(CONFIG['seed'])
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DATA_ROOT = Path(CONFIG['data_root'])
ROI_METADATA_PATH = Path(CONFIG['roi_metadata_path'])
SELECTION_METADATA_PATH = Path(CONFIG['selection_metadata_path'])
RESULTS_ROOT = Path(CONFIG['results_root'])

resume_run_id = CONFIG.get('resume_run_id')
if resume_run_id:
    RUN_ID = str(resume_run_id)
else:
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M') + f"_eyebrow_hog_lbp_svm_seed{SEED}"

RUN_DIR = RESULTS_ROOT / RUN_ID
if RUN_DIR.exists() and not resume_run_id:
    raise FileExistsError(
        f'Bu run_id zaten mevcut: {RUN_DIR}. Aynı dizine iki farklı deney yazılmaz. '
        'Yeni bir dakika bekleyin veya resume_run_id ile bilinçli olarak devam edin.'
    )
DIRS = {
    'checkpoints': RUN_DIR / 'checkpoints',
    'logs': RUN_DIR / 'logs',
    'metrics': RUN_DIR / 'metrics',
    'predictions': RUN_DIR / 'predictions',
    'figures': RUN_DIR / 'figures',
    'artifacts': RUN_DIR / 'artifacts',
}
for path in [RUN_DIR, *DIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)

# Kaynak veriye yazılmadığını garanti eden temel yol kontrolü
assert DATA_ROOT.exists(), f'Data root bulunamadı: {DATA_ROOT}'
assert ROI_METADATA_PATH.exists(), f'ROI metadata bulunamadı: {ROI_METADATA_PATH}'
assert SELECTION_METADATA_PATH.exists(), f'Seçim metadata bulunamadı: {SELECTION_METADATA_PATH}'
assert RUN_DIR.resolve() != DATA_ROOT.resolve()
assert DATA_ROOT.resolve() not in RUN_DIR.resolve().parents

# Logger
LOG_PATH = DIRS['logs'] / 'run.log'
logger = logging.getLogger('hog_lbp_svm')
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler = logging.FileHandler(LOG_PATH, encoding='utf-8')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json_atomic(data, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    with temp.open('w', encoding='utf-8') as f:
        json.dump(json_safe(data), f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    # Doğrulama
    with temp.open('r', encoding='utf-8') as f:
        json.load(f)
    os.replace(temp, target)


def write_csv_atomic(df: pd.DataFrame, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    df.to_csv(temp, index=False)
    _ = pd.read_csv(temp)
    os.replace(temp, target)


def save_npz_atomic(target: Path, **arrays):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    with temp.open('wb') as f:
        np.savez_compressed(f, **arrays)
        f.flush()
        os.fsync(f.fileno())
    with np.load(temp, allow_pickle=False) as z:
        if not z.files:
            raise RuntimeError(f'NPZ doğrulama başarısız: {temp}')
    os.replace(temp, target)


def atomic_joblib_dump(obj, target: Path, required_keys=None):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + '.tmp')
    joblib.dump(obj, temp)
    loaded = joblib.load(temp)
    if required_keys:
        if not isinstance(loaded, dict) or not set(required_keys).issubset(loaded.keys()):
            raise RuntimeError(f'Checkpoint doğrulama başarısız: {temp}')
    os.replace(temp, target)


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def save_figure(fig, target: Path):
    fig.tight_layout()
    fig.savefig(target, dpi=int(CONFIG['figure_dpi']), bbox_inches='tight')
    plt.close(fig)
    with Image.open(target) as img:
        width, height = img.size
    min_edge = int(CONFIG['minimum_figure_short_edge_px'])
    assert min(width, height) >= min_edge, f'Figure resolution too low: {(width, height)}'
    return width, height


resolved_config = dict(CONFIG)
resolved_config.update({
    'run_id': RUN_ID,
    'run_dir': str(RUN_DIR),
    'created_at': datetime.now().isoformat(),
})
with (RUN_DIR / 'config_resolved.yaml').open('w', encoding='utf-8') as f:
    yaml.safe_dump(resolved_config, f, sort_keys=False, allow_unicode=True)

logger.info('RUN_ID=%s', RUN_ID)
logger.info('Input=%s', DATA_ROOT)
logger.info('Output=%s', RUN_DIR)


2026-08-07 09:48:40,578 | INFO | RUN_ID=20260807_0948_eyebrow_hog_lbp_svm_seed42


INFO:hog_lbp_svm:RUN_ID=20260807_0948_eyebrow_hog_lbp_svm_seed42


2026-08-07 09:48:40,590 | INFO | Input=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş


INFO:hog_lbp_svm:Input=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş


2026-08-07 09:48:40,596 | INFO | Output=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260807_0948_eyebrow_hog_lbp_svm_seed42


INFO:hog_lbp_svm:Output=/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260807_0948_eyebrow_hog_lbp_svm_seed42


In [4]:

# 4) Metadata oluşturma + veri muhasebesi + split leakage quality gate
roi = pd.read_csv(ROI_METADATA_PATH)
selection = pd.read_csv(SELECTION_METADATA_PATH, encoding='utf-8-sig')

required_roi_cols = {
    'sample_id', 'frame_index', 'face_index', 'label', 'split', 'status',
    'skip_reason', 'output_path', 'input_relative_path', 'input_sha256',
    'output_sha256', 'run_id'
}
required_selection_cols = {'sinif', 'split', 'orijinal_yol', 'yeni_yol', 'dosya_adi'}

missing_roi = required_roi_cols - set(roi.columns)
missing_selection = required_selection_cols - set(selection.columns)
assert not missing_roi, f'ROI metadata eksik sütunlar: {sorted(missing_roi)}'
assert not missing_selection, f'Seçim metadata eksik sütunlar: {sorted(missing_selection)}'

# Kaynak seçim dosyasıyla deterministic join.
# Not: Kaş metadata içindeki source_video alanı boş olduğundan burada tahmin yürütülmez;
# seçim metadata'sındaki gerçek orijinal yoldan kaynak klasör adı geri kazanılır.
selection = selection.copy()
selection['dosya_adi'] = selection['dosya_adi'].astype(str)
assert selection['dosya_adi'].is_unique, 'secim_metadata.csv içinde dosya_adi benzersiz değil.'
selection['source_video_resolved'] = selection['orijinal_yol'].map(lambda p: Path(str(p)).parent.name)
selection['original_frame_index'] = selection['orijinal_yol'].map(
    lambda p: int(Path(str(p)).stem.split('_')[-1])
)

roi = roi.copy()
roi['dosya_adi'] = roi['input_relative_path'].map(lambda p: Path(str(p)).name)
merged = roi.merge(
    selection[['dosya_adi', 'source_video_resolved', 'original_frame_index', 'orijinal_yol']],
    on='dosya_adi',
    how='left',
    validate='one_to_one',
)
assert merged['source_video_resolved'].notna().all(), 'Bazı kaş frameleri secim_metadata.csv ile eşleşmedi.'

# Ekip standardındaki metadata şemasını eğitim için standardize et.
metadata = pd.DataFrame({
    'sample_id': merged['sample_id'].astype(str),
    'source_video': merged['source_video_resolved'].astype(str),
    'frame_index': merged['original_frame_index'].astype(int),
    'face_index': merged['face_index'].fillna(0).astype(int),
    'roi_state': 'not_applicable_eyebrow',
    'label': merged['label'].astype(str).str.lower(),
    'split': merged['split'].astype(str).str.lower(),
    'status': merged['status'].astype(str).str.upper(),
    'skip_reason': merged['skip_reason'].fillna('').astype(str),
    'sha256': np.where(
        merged['output_sha256'].notna() & (merged['output_sha256'].astype(str) != ''),
        merged['output_sha256'].astype(str),
        merged['input_sha256'].astype(str),
    ),
    'output_path': merged['output_path'].astype(str),
    'run_id': merged['run_id'].astype(str),  # ROI verisini üreten run
    'training_run_id': RUN_ID,
    'original_path': merged['orijinal_yol'].astype(str),
    'selected_frame_name': merged['dosya_adi'].astype(str),
})

assert set(metadata['label'].unique()) == {'real', 'fake'}, metadata['label'].value_counts().to_dict()
assert set(metadata['split'].unique()) == {'train', 'val', 'test'}, metadata['split'].value_counts().to_dict()
assert set(metadata['status'].unique()).issubset({'SUCCESS', 'SKIPPED', 'ERROR'})

# Accounting equality
total_inputs = len(metadata)
success_count = int((metadata['status'] == 'SUCCESS').sum())
skipped_count = int((metadata['status'] == 'SKIPPED').sum())
error_count = int((metadata['status'] == 'ERROR').sum())
assert total_inputs == success_count + skipped_count + error_count, 'Girdi-çıktı sayıları uyuşmuyor!'
assert metadata['sample_id'].is_unique, 'Mükerrer sample_id tespit edildi!'
assert metadata['output_path'].notna().all(), 'Eksik output_path var!'

success_mask = metadata['status'] == 'SUCCESS'
missing_success_files = [p for p in metadata.loc[success_mask, 'output_path'] if not Path(p).exists()]
assert not missing_success_files, f'SUCCESS statülü {len(missing_success_files)} dosya diskte bulunamadı.'

eligible = metadata.loc[success_mask].copy().reset_index(drop=True)
assert eligible['source_video'].notna().all()

# Video düzeyinde split kesişimi kesinlikle sıfır olmalı.
video_sets = {
    split: set(eligible.loc[eligible['split'] == split, 'source_video'].astype(str))
    for split in ['train', 'val', 'test']
}
intersections = {
    'train_val': len(video_sets['train'] & video_sets['val']),
    'train_test': len(video_sets['train'] & video_sets['test']),
    'val_test': len(video_sets['val'] & video_sets['test']),
}
assert intersections['train_val'] == 0, 'Train ve Val arasında video sızıntısı var!'
assert intersections['train_test'] == 0, 'Train ve Test arasında video sızıntısı var!'
assert intersections['val_test'] == 0, 'Val ve Test arasında video sızıntısı var!'

# Her source_video tek label/split ile temsil edilmeli.
assert eligible.groupby('source_video')['split'].nunique().max() == 1, 'Aynı source_video birden fazla splitte!'
assert eligible.groupby('source_video')['label'].nunique().max() == 1, 'Aynı source_video birden fazla label ile eşleşiyor!'

split_class_counts = (
    eligible.groupby(['split', 'label']).size().unstack(fill_value=0).to_dict(orient='index')
)
split_source_video_counts = {
    split: int(eligible.loc[eligible['split'] == split, 'source_video'].nunique())
    for split in ['train', 'val', 'test']
}

accounting = {
    'run_id': RUN_ID,
    'total_inputs': total_inputs,
    'success_count': success_count,
    'skipped_count': skipped_count,
    'error_count': error_count,
    'training_eligible_success_count': len(eligible),
    'unique_source_videos': int(eligible['source_video'].nunique()),
    'split_class_counts': split_class_counts,
    'split_source_video_counts': split_source_video_counts,
    'source_video_intersections': intersections,
}
write_json_atomic(accounting, DIRS['metrics'] / 'data_accounting.json')
write_csv_atomic(metadata, DIRS['artifacts'] / 'metadata_used.csv')

print(json.dumps(accounting, ensure_ascii=False, indent=2))


{
  "run_id": "20260807_0948_eyebrow_hog_lbp_svm_seed42",
  "total_inputs": 3000,
  "success_count": 1962,
  "skipped_count": 1038,
  "error_count": 0,
  "training_eligible_success_count": 1962,
  "unique_source_videos": 1429,
  "split_class_counts": {
    "test": {
      "fake": 95,
      "real": 101
    },
    "train": {
      "fake": 776,
      "real": 786
    },
    "val": {
      "fake": 98,
      "real": 106
    }
  },
  "split_source_video_counts": {
    "train": 1145,
    "val": 146,
    "test": 138
  },
  "source_video_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  }
}


In [5]:

# 5) HOG + LBP feature extraction fonksiyonları ve klasik-ML smoke test
IMG_W = int(CONFIG['image_width'])
IMG_H = int(CONFIG['image_height'])
HOG_CFG = CONFIG['hog']
LBP_CFG = CONFIG['lbp']


def load_preprocessed_grayscale(path: str) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Image could not be read: {path}')
    image = cv2.resize(image, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    return image


def extract_hog(gray: np.ndarray) -> np.ndarray:
    feat = hog(
        gray,
        orientations=int(HOG_CFG['orientations']),
        pixels_per_cell=tuple(HOG_CFG['pixels_per_cell']),
        cells_per_block=tuple(HOG_CFG['cells_per_block']),
        block_norm=str(HOG_CFG['block_norm']),
        feature_vector=True,
    )
    return np.asarray(feat, dtype=np.float32)


def extract_lbp(gray: np.ndarray) -> np.ndarray:
    points = int(LBP_CFG['points'])
    radius = int(LBP_CFG['radius'])
    method = str(LBP_CFG['method'])
    lbp_img = local_binary_pattern(gray, P=points, R=radius, method=method)
    # uniform LBP için P + 2 olası bin
    n_bins = points + 2 if method == 'uniform' else int(lbp_img.max() + 1)
    hist, _ = np.histogram(lbp_img.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
    hist = hist.astype(np.float32)
    hist /= hist.sum() + 1e-12
    return hist


def extract_features(path: str):
    gray = load_preprocessed_grayscale(path)
    hog_feat = extract_hog(gray)
    lbp_feat = extract_lbp(gray)
    fused = np.concatenate([hog_feat, lbp_feat]).astype(np.float32)
    if not np.isfinite(fused).all():
        raise FloatingPointError(f'NaN/Inf feature detected: {path}')
    return hog_feat, lbp_feat, fused


# Smoke gate: train setinden iki sınıfı içeren en az 4 örnekte extraction + fit + inference.
smoke_rows = pd.concat([
    eligible[(eligible['split'] == 'train') & (eligible['label'] == 'real')].head(2),
    eligible[(eligible['split'] == 'train') & (eligible['label'] == 'fake')].head(2),
], ignore_index=True)
assert len(smoke_rows) == 4 and smoke_rows['label'].nunique() == 2

smoke_X = []
smoke_y = []
for row in smoke_rows.itertuples(index=False):
    _, _, feat = extract_features(row.output_path)
    smoke_X.append(feat)
    smoke_y.append(1 if row.label == CONFIG['positive_class'] else 0)
smoke_X = np.vstack(smoke_X)
smoke_y = np.asarray(smoke_y)
smoke_scaler = StandardScaler().fit(smoke_X)
smoke_model = SVC(kernel='linear', C=1.0, class_weight='balanced')
smoke_model.fit(smoke_scaler.transform(smoke_X), smoke_y)
smoke_pred = smoke_model.predict(smoke_scaler.transform(smoke_X))
assert len(smoke_pred) == len(smoke_y)
assert np.isfinite(smoke_X).all()

print('Classical-ML smoke test: PASS')
print('Feature dimension:', smoke_X.shape[1])


Classical-ML smoke test: PASS
Feature dimension: 3790


In [7]:

# 6) Tüm splitler için feature extraction (split bazlı atomik cache)

def build_split_features(split: str):
    cache_path = DIRS['artifacts'] / f'features_{split}.npz'
    if cache_path.exists():
        logger.info('Cached features loaded: %s', cache_path)
        with np.load(cache_path, allow_pickle=False) as z:
            return {
                'X_hog': z['X_hog'],
                'X_lbp': z['X_lbp'],
                'X_fused': z['X_fused'],
                'y': z['y'],
                'sample_id': z['sample_id'].astype(str),
                'source_video': z['source_video'].astype(str),
                'frame_index': z['frame_index'],
                'output_path': z['output_path'].astype(str),
            }

    part = eligible[eligible['split'] == split].copy().reset_index(drop=True)
    hog_rows, lbp_rows, fused_rows = [], [], []
    labels = []

    start = time.time()
    for i, row in enumerate(part.itertuples(index=False), start=1):
        h, l, f = extract_features(row.output_path)
        hog_rows.append(h)
        lbp_rows.append(l)
        fused_rows.append(f)
        labels.append(1 if row.label == CONFIG['positive_class'] else 0)
        if i % 100 == 0 or i == len(part):
            logger.info('Feature extraction %s: %d/%d', split, i, len(part))

    data = {
        'X_hog': np.vstack(hog_rows).astype(np.float32),
        'X_lbp': np.vstack(lbp_rows).astype(np.float32),
        'X_fused': np.vstack(fused_rows).astype(np.float32),
        'y': np.asarray(labels, dtype=np.int64),
        'sample_id': part['sample_id'].astype(str).to_numpy(dtype=str),
        'source_video': part['source_video'].astype(str).to_numpy(dtype=str),
        'frame_index': part['frame_index'].to_numpy(dtype=np.int64),
        'output_path': part['output_path'].astype(str).to_numpy(dtype=str),
    }

    assert len(data['X_fused']) == len(part)
    assert np.isfinite(data['X_fused']).all()
    save_npz_atomic(cache_path, **data)
    logger.info('Feature extraction %s completed in %.2fs', split, time.time() - start)
    return data


features = {split: build_split_features(split) for split in ['train', 'val', 'test']}

hog_dim = int(features['train']['X_hog'].shape[1])
lbp_dim = int(features['train']['X_lbp'].shape[1])
fused_dim = int(features['train']['X_fused'].shape[1])
assert fused_dim == hog_dim + lbp_dim
for split in ['train', 'val', 'test']:
    assert features[split]['X_hog'].shape[1] == hog_dim
    assert features[split]['X_lbp'].shape[1] == lbp_dim
    assert features[split]['X_fused'].shape[1] == fused_dim

feature_dimensions = {
    'hog_dim': hog_dim,
    'lbp_dim': lbp_dim,
    'fused_dim': fused_dim,
    'image_size': [IMG_W, IMG_H],
}
write_json_atomic(feature_dimensions, DIRS['artifacts'] / 'feature_dimensions.json')

feature_pipeline = {
    'input': 'eyebrow ROI',
    'preprocessing': ['grayscale', f'resize_{IMG_W}x{IMG_H}'],
    'features': {
        'HOG': HOG_CFG,
        'LBP': LBP_CFG,
        'fusion': 'concatenation',
    },
    'normalization': 'StandardScaler fit on train only',
    'classifier': 'SVM selected on validation only',
    'pretrained_model_used': False,
}
write_json_atomic(feature_pipeline, DIRS['artifacts'] / 'feature_pipeline.json')

print(json.dumps(feature_dimensions, indent=2))


2026-08-07 09:51:03,470 | INFO | Feature extraction train: 100/1562


INFO:hog_lbp_svm:Feature extraction train: 100/1562


2026-08-07 09:52:27,121 | INFO | Feature extraction train: 200/1562


INFO:hog_lbp_svm:Feature extraction train: 200/1562


2026-08-07 09:53:51,635 | INFO | Feature extraction train: 300/1562


...

2026-08-07 10:17:19,166 | INFO | Feature extraction test: 196/196


INFO:hog_lbp_svm:Feature extraction test: 196/196


2026-08-07 10:17:19,613 | INFO | Feature extraction test completed in 165.73s


INFO:hog_lbp_svm:Feature extraction test completed in 165.73s


{
  "hog_dim": 3780,
  "lbp_dim": 10,
  "fused_dim": 3790,
  "image_size": [
    128,
    64
  ]
}


In [ ]:

# 7) Train-only scaling + validation-only SVM model/threshold selection
X_train = features['train']['X_fused']
y_train = features['train']['y']
X_val = features['val']['X_fused']
y_val = features['val']['y']
X_test = features['test']['X_fused']
y_test = features['test']['y']

# Normalizasyon yalnızca train üzerinde öğrenilir.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
# Test seti bu aşamada dönüştürülmez/değerlendirilmez.

atomic_joblib_dump(scaler, DIRS['artifacts'] / 'feature_preprocessing.joblib')


def metrics_from_scores(y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else float('nan')
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'specificity': specificity,
        'roc_auc': roc_auc_score(y_true, scores),
        'average_precision': average_precision_score(y_true, scores),
    }


def best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if len(thresholds) == 0:
        return 0.0
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    idx = int(np.nanargmax(f1_values))
    return float(thresholds[idx])


svm_cfg = CONFIG['svm']
candidates = []
for kernel in svm_cfg['kernels']:
    for C in svm_cfg['c_values']:
        if kernel == 'linear':
            candidates.append({'kernel': 'linear', 'C': float(C), 'gamma': None})
        elif kernel == 'rbf':
            for gamma in svm_cfg['gamma_values']:
                candidates.append({'kernel': 'rbf', 'C': float(C), 'gamma': gamma})
        else:
            raise ValueError(f'Unsupported kernel: {kernel}')

search_rows = []
search_start = time.time()
for idx, params in enumerate(candidates, start=1):
    kwargs = {
        'kernel': params['kernel'],
        'C': params['C'],
        'class_weight': svm_cfg['class_weight'],
        'probability': False,
        'cache_size': 2048,
    }
    if params['kernel'] == 'rbf':
        kwargs['gamma'] = params['gamma']

    model = SVC(**kwargs)
    t0 = time.time()
    model.fit(X_train_s, y_train)
    val_scores = model.decision_function(X_val_s)
    assert np.isfinite(val_scores).all(), 'Validation decision scores contain NaN/Inf.'
    threshold = best_f1_threshold(y_val, val_scores)
    m = metrics_from_scores(y_val, val_scores, threshold)
    row = {
        **params,
        'threshold': threshold,
        **{f'val_{k}': float(v) for k, v in m.items()},
        'fit_seconds': time.time() - t0,
    }
    search_rows.append(row)
    write_csv_atomic(pd.DataFrame(search_rows), DIRS['metrics'] / 'svm_validation_search.csv')
    logger.info('SVM candidate %d/%d: %s | val_f1=%.4f', idx, len(candidates), params, m['f1'])

search_df = pd.DataFrame(search_rows)
# Birincil seçim: val F1. Eşitlikte ROC-AUC, sonra accuracy.
search_df = search_df.sort_values(
    ['val_f1', 'val_roc_auc', 'val_accuracy'],
    ascending=[False, False, False],
).reset_index(drop=True)
best_row = search_df.iloc[0].to_dict()

best_kwargs = {
    'kernel': best_row['kernel'],
    'C': float(best_row['C']),
    'class_weight': svm_cfg['class_weight'],
    'probability': False,
    'cache_size': 2048,
}
if best_row['kernel'] == 'rbf':
    best_kwargs['gamma'] = best_row['gamma']

# Seçim validation ile tamamlandıktan sonra nihai model SADECE train üzerinde aynı parametrelerle fit edilir.
best_model = SVC(**best_kwargs)
best_model.fit(X_train_s, y_train)
selected_threshold = float(best_row['threshold'])

selection_summary = {
    'selection_set': 'validation',
    'selection_metric': svm_cfg['selection_metric'],
    'kernel': best_row['kernel'],
    'C': float(best_row['C']),
    'gamma': None if pd.isna(best_row.get('gamma')) else best_row.get('gamma'),
    'threshold': selected_threshold,
    'validation_metrics': {k.replace('val_', ''): float(v) for k, v in best_row.items() if str(k).startswith('val_')},
    'candidate_count': len(candidates),
    'search_seconds': time.time() - search_start,
}
write_json_atomic(selection_summary, DIRS['metrics'] / 'svm_selection.json')

# Standalone model artifact
model_artifact = {
    'model': best_model,
    'scaler': scaler,
    'threshold': selected_threshold,
    'positive_class': CONFIG['positive_class'],
    'feature_pipeline': feature_pipeline,
    'config': resolved_config,
}
atomic_joblib_dump(model_artifact, DIRS['artifacts'] / 'svm_model.joblib', required_keys=['model', 'scaler', 'threshold'])

print(json.dumps(json_safe(selection_summary), ensure_ascii=False, indent=2))


In [ ]:

# 8) Atomik klasik-ML checkpoint + fresh-load inference quality gates
checkpoint_state = {
    'checkpoint_schema': 'classical_ml_v1',
    'training_stage': 'completed_svm_fit',
    'epoch': None,
    'model': best_model,
    'preprocessor': scaler,
    'threshold': selected_threshold,
    'config': resolved_config,
    'best_metric_score': float(best_row['val_f1']),
    'python_random_state': random.getstate(),
    'numpy_random_state': np.random.get_state(),
    'non_applicable_pytorch_fields': {
        'model_state_dict': 'N/A - sklearn SVC object serialized as a whole',
        'optimizer_state_dict': 'N/A - SVC has no PyTorch optimizer',
        'scheduler_state_dict': 'N/A',
        'scaler_state_dict_mixed_precision': 'N/A',
        'forward_backward_smoke_test': 'N/A - classical ML; equivalent fit/predict smoke test used',
    },
}

for name in ['best.ckpt', 'last.ckpt', 'checkpoint_quality_gate.ckpt']:
    atomic_joblib_dump(
        checkpoint_state,
        DIRS['checkpoints'] / name,
        required_keys=['checkpoint_schema', 'model', 'preprocessor', 'threshold', 'config'],
    )

# Checkpoint reload: aynı validation skorlarını üretmeli.
reloaded_ckpt = joblib.load(DIRS['checkpoints'] / 'best.ckpt')
reload_val_scores = reloaded_ckpt['model'].decision_function(
    reloaded_ckpt['preprocessor'].transform(X_val)
)
original_val_scores = best_model.decision_function(X_val_s)
max_abs_diff = float(np.max(np.abs(reload_val_scores - original_val_scores)))
assert np.allclose(reload_val_scores, original_val_scores, rtol=1e-10, atol=1e-12)

# Fresh-load inference: bağımsız svm_model.joblib dosyası sıfırdan yüklenerek tahmin üretmeli.
fresh = joblib.load(DIRS['artifacts'] / 'svm_model.joblib')
probe_X = X_val[: min(8, len(X_val))]
probe_scores = fresh['model'].decision_function(fresh['scaler'].transform(probe_X))
probe_pred = (probe_scores >= fresh['threshold']).astype(int)
assert len(probe_pred) == len(probe_X)
assert np.isfinite(probe_scores).all()

fresh_load_test = {
    'status': 'PASS',
    'probe_count': len(probe_X),
    'predictions': probe_pred.tolist(),
    'checkpoint_reload_max_abs_score_diff': max_abs_diff,
}
write_json_atomic(fresh_load_test, DIRS['metrics'] / 'fresh_load_inference_test.json')

quality_gates = {
    'schema_test': 'PASS',
    'split_test': 'PASS',
    'data_accounting_test': 'PASS',
    'numerical_feature_test': 'PASS',
    'classical_ml_smoke_test': 'PASS',
    'pytorch_forward_backward_smoke_test': 'NOT_APPLICABLE_CLASSICAL_ML',
    'checkpoint_atomic_save_reload_test': 'PASS',
    'fresh_load_inference_test': 'PASS',
    'test_set_used_for_selection': False,
}
write_json_atomic(quality_gates, DIRS['metrics'] / 'quality_gates.json')
print(json.dumps(quality_gates, indent=2))


In [ ]:

# 9) Nihai TEST değerlendirmesi — test setine ilk kez burada bakılır
# Bu hücreye kadar hiçbir test metriği model/threshold seçimi için kullanılmadı.

def build_prediction_df(split_name, split_features, scores, threshold):
    y_true = split_features['y']
    y_pred = (scores >= threshold).astype(int)
    return pd.DataFrame({
        'sample_id': split_features['sample_id'],
        'source_video': split_features['source_video'],
        'frame_index': split_features['frame_index'],
        'split': split_name,
        'y_true': y_true,
        'label': np.where(y_true == 1, CONFIG['positive_class'], 'real'),
        'decision_score': scores,
        'threshold': threshold,
        'y_pred': y_pred,
        'predicted_label': np.where(y_pred == 1, CONFIG['positive_class'], 'real'),
        'correct': y_true == y_pred,
        'output_path': split_features['output_path'],
        'run_id': RUN_ID,
    })


def aggregate_video_level(frame_df: pd.DataFrame):
    label_nunique = frame_df.groupby('source_video')['y_true'].nunique()
    assert label_nunique.max() == 1, 'A source_video has conflicting labels.'
    video = frame_df.groupby('source_video', as_index=False).agg(
        y_true=('y_true', 'first'),
        decision_score=('decision_score', 'mean'),
        frame_count=('sample_id', 'size'),
    )
    video['threshold'] = selected_threshold
    video['y_pred'] = (video['decision_score'] >= selected_threshold).astype(int)
    video['label'] = np.where(video['y_true'] == 1, CONFIG['positive_class'], 'real')
    video['predicted_label'] = np.where(video['y_pred'] == 1, CONFIG['positive_class'], 'real')
    video['correct'] = video['y_true'] == video['y_pred']
    video['run_id'] = RUN_ID
    return video


X_test_s = scaler.transform(X_test)
test_scores = best_model.decision_function(X_test_s)
assert np.isfinite(test_scores).all()

frame_pred = build_prediction_df('test', features['test'], test_scores, selected_threshold)
video_pred = aggregate_video_level(frame_pred)

frame_metrics = metrics_from_scores(frame_pred['y_true'].to_numpy(), frame_pred['decision_score'].to_numpy(), selected_threshold)
video_metrics = metrics_from_scores(video_pred['y_true'].to_numpy(), video_pred['decision_score'].to_numpy(), selected_threshold)

write_csv_atomic(frame_pred, DIRS['predictions'] / 'test_predictions_frame_level.csv')
write_csv_atomic(video_pred, DIRS['predictions'] / 'test_predictions_video_level.csv')
write_json_atomic(frame_metrics, DIRS['metrics'] / 'test_metrics_frame_level.json')
write_json_atomic(video_metrics, DIRS['metrics'] / 'test_metrics_video_level.json')

metrics_summary = pd.DataFrame([
    {'level': 'frame', **frame_metrics},
    {'level': 'video', **video_metrics},
])
write_csv_atomic(metrics_summary, DIRS['metrics'] / 'test_metrics_summary.csv')

print(metrics_summary.to_string(index=False))


In [ ]:

# 10) İngilizce ve yüksek çözünürlüklü rapor grafikleri
figure_records = []


def register_figure(fig, filename):
    path = DIRS['figures'] / filename
    width, height = save_figure(fig, path)
    figure_records.append({
        'file': filename,
        'width_px': width,
        'height_px': height,
        'short_edge_px': min(width, height),
        'quality_pass': min(width, height) >= int(CONFIG['minimum_figure_short_edge_px']),
    })


# Dataset distribution
count_df = eligible.groupby(['split', 'label']).size().unstack(fill_value=0).reindex(['train', 'val', 'test'])
fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
x = np.arange(len(count_df.index))
width = 0.35
ax.bar(x - width/2, count_df.get('real', pd.Series(0, index=count_df.index)), width, label='Real')
ax.bar(x + width/2, count_df.get('fake', pd.Series(0, index=count_df.index)), width, label='Fake')
ax.set_title('Dataset Distribution by Split and Class', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Split', fontsize=11)
ax.set_ylabel('Number of Frames', fontsize=11)
ax.set_xticks(x, ['Train', 'Validation', 'Test'])
ax.legend(frameon=True)
ax.grid(True, axis='y', alpha=0.25)
register_figure(fig, 'dataset_distribution.png')

# Validation candidate comparison
plot_search = search_df.copy().sort_values('val_f1', ascending=False).reset_index(drop=True)
labels = [
    f"{r.kernel} | C={r.C}" + (f" | g={r.gamma}" if r.kernel == 'rbf' else '')
    for r in plot_search.itertuples(index=False)
]
fig, ax = plt.subplots(figsize=(12, 7), dpi=int(CONFIG['figure_dpi']))
ax.bar(np.arange(len(plot_search)), plot_search['val_f1'])
ax.set_title('Validation F1 by SVM Candidate', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('SVM Candidate', fontsize=11)
ax.set_ylabel('Validation F1', fontsize=11)
ax.set_xticks(np.arange(len(labels)), labels, rotation=60, ha='right')
ax.grid(True, axis='y', alpha=0.25)
register_figure(fig, 'svm_validation_f1_comparison.png')


def confusion_plot(df, level_name, filename):
    cm = confusion_matrix(df['y_true'], df['y_pred'], labels=[0, 1])
    fig, ax = plt.subplots(figsize=(8, 8), dpi=int(CONFIG['figure_dpi']))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Real', 'Fake'])
    disp.plot(ax=ax, colorbar=False, values_format='d')
    ax.set_title(f'Test Confusion Matrix — {level_name}', fontsize=14, fontweight='bold', pad=12)
    register_figure(fig, filename)


def roc_plot(df, level_name, filename):
    fpr, tpr, _ = roc_curve(df['y_true'], df['decision_score'])
    auc_value = roc_auc_score(df['y_true'], df['decision_score'])
    fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
    ax.plot(fpr, tpr, linewidth=2, label=f'ROC AUC = {auc_value:.3f}')
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
    ax.set_title(f'Test ROC Curve — {level_name}', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.legend(frameon=True, loc='lower right')
    ax.grid(True, alpha=0.25)
    register_figure(fig, filename)


def pr_plot(df, level_name, filename):
    precision, recall, _ = precision_recall_curve(df['y_true'], df['decision_score'])
    ap = average_precision_score(df['y_true'], df['decision_score'])
    fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
    ax.plot(recall, precision, linewidth=2, label=f'Average Precision = {ap:.3f}')
    ax.set_title(f'Test Precision-Recall Curve — {level_name}', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Recall', fontsize=11)
    ax.set_ylabel('Precision', fontsize=11)
    ax.legend(frameon=True, loc='lower left')
    ax.grid(True, alpha=0.25)
    register_figure(fig, filename)


confusion_plot(frame_pred, 'Frame Level', 'test_confusion_matrix_frame_level.png')
confusion_plot(video_pred, 'Video Level', 'test_confusion_matrix_video_level.png')
roc_plot(frame_pred, 'Frame Level', 'test_roc_curve_frame_level.png')
roc_plot(video_pred, 'Video Level', 'test_roc_curve_video_level.png')
pr_plot(frame_pred, 'Frame Level', 'test_precision_recall_curve_frame_level.png')
pr_plot(video_pred, 'Video Level', 'test_precision_recall_curve_video_level.png')

# Decision score distribution
fig, ax = plt.subplots(figsize=(10, 6), dpi=int(CONFIG['figure_dpi']))
real_scores = frame_pred.loc[frame_pred['y_true'] == 0, 'decision_score']
fake_scores = frame_pred.loc[frame_pred['y_true'] == 1, 'decision_score']
ax.hist(real_scores, bins=30, alpha=0.65, label='Real')
ax.hist(fake_scores, bins=30, alpha=0.65, label='Fake')
ax.axvline(selected_threshold, linestyle='--', linewidth=2, label=f'Threshold = {selected_threshold:.3f}')
ax.set_title('Test Decision Score Distribution — Frame Level', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('SVM Decision Score', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.legend(frameon=True)
ax.grid(True, alpha=0.25)
register_figure(fig, 'test_decision_score_distribution.png')

figure_audit = pd.DataFrame(figure_records)
assert figure_audit['quality_pass'].all(), 'At least one figure violates the minimum resolution rule.'
write_csv_atomic(figure_audit, DIRS['metrics'] / 'figure_quality_audit.csv')
figure_audit


In [ ]:

# 11) Environment lock, output manifest ve run summary
# Gerçek Colab ortamı deney sonunda kilitlenir.
requirements_text = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'requirements_lock.txt').write_text(requirements_text, encoding='utf-8')

environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': __import__('sklearn').__version__,
    'scikit_image': __import__('skimage').__version__,
    'run_id': RUN_ID,
}
write_json_atomic(environment, RUN_DIR / 'environment.json')

run_summary = {
    'run_id': RUN_ID,
    'status': 'COMPLETED',
    'model': {
        'pretrained_weights': False,
        'feature_fusion': ['HOG', 'LBP'],
        'classifier': f"{selection_summary['kernel'].upper()} SVM",
        'C': selection_summary['C'],
        'gamma': selection_summary['gamma'],
        'threshold': selection_summary['threshold'],
    },
    'paths': {
        'input_data': str(DATA_ROOT),
        'roi_metadata': str(ROI_METADATA_PATH),
        'selection_metadata': str(SELECTION_METADATA_PATH),
        'output_run': str(RUN_DIR),
        'best_checkpoint': str(DIRS['checkpoints'] / 'best.ckpt'),
        'svm_model': str(DIRS['artifacts'] / 'svm_model.joblib'),
    },
    'data': accounting,
    'feature_dimensions': feature_dimensions,
    'selected_svm': selection_summary,
    'test_metrics': {
        'frame_level': frame_metrics,
        'video_level': video_metrics,
    },
    'quality_gates': quality_gates,
    'figure_count': len(figure_records),
    'checkpoint_note': (
        'PyTorch optimizer/epoch state is not applicable to HOG+LBP+sklearn SVC. '
        'The complete SVC, train-fitted StandardScaler, threshold, config and RNG states '
        'are atomically serialized and reload-tested.'
    ),
    'completed_at': datetime.now().isoformat(),
}
write_json_atomic(run_summary, RUN_DIR / 'run_summary.json')

# Manifest kendisini hash'leyemez (self-referential), bu nedenle output_manifest.csv kendisini bilinçli olarak dışlar.
manifest_rows = []
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file() and path.name != 'output_manifest.csv' and not path.name.endswith('.tmp'):
        manifest_rows.append({
            'relative_path': str(path.relative_to(RUN_DIR)),
            'size_bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        })
manifest = pd.DataFrame(manifest_rows)
write_csv_atomic(manifest, RUN_DIR / 'output_manifest.csv')

# Son kontrol: geçici dosya kalmamalı.
tmp_files = [str(p) for p in RUN_DIR.rglob('*.tmp')]
assert not tmp_files, f'Temporary files remain: {tmp_files}'

print('=' * 80)
print('EXPERIMENT COMPLETED')
print('Run ID:', RUN_ID)
print('Output:', RUN_DIR)
print('\nFrame-level test metrics:')
print(json.dumps(json_safe(frame_metrics), indent=2))
print('\nVideo-level test metrics:')
print(json.dumps(json_safe(video_metrics), indent=2))
print('\nGenerated output files:', len(manifest))
print('=' * 80)
